<a href="https://colab.research.google.com/github/KaevienAsoran/llm-from-foundations/blob/main/week01/week01_mini_word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

In [2]:
corpus = [
    "the cat likes milk",
    "the dog likes milk",
    "the cat runs outside",
    "the dog runs outside",
    "my cat is friendly",
    "my dog is friendly",
    "the cat eats food",
    "the dog eats food"
]

tokenized_corpus = [
    sentence.split()
    for sentence in corpus
]

print(tokenized_corpus)

[['the', 'cat', 'likes', 'milk'], ['the', 'dog', 'likes', 'milk'], ['the', 'cat', 'runs', 'outside'], ['the', 'dog', 'runs', 'outside'], ['my', 'cat', 'is', 'friendly'], ['my', 'dog', 'is', 'friendly'], ['the', 'cat', 'eats', 'food'], ['the', 'dog', 'eats', 'food']]


In [3]:
all_words = []

for sentence in tokenized_corpus:
    all_words.extend(sentence)

vocab = sorted(set(all_words))

word_to_id = {
    word: i
    for i, word in enumerate(vocab)
}

id_to_word = {
    i: word
    for word, i in word_to_id.items()
}

print("Vocabulary size:", len(vocab))
print(word_to_id)

Vocabulary size: 12
{'cat': 0, 'dog': 1, 'eats': 2, 'food': 3, 'friendly': 4, 'is': 5, 'likes': 6, 'milk': 7, 'my': 8, 'outside': 9, 'runs': 10, 'the': 11}


让模型通过中间的词来预测左右两侧的词可能是什么

In [4]:
window_size = 1
training_pairs = []

for sentence in tokenized_corpus:

    for center_index in range(len(sentence)):

        center_word = sentence[center_index]

        left = max(
            0,
            center_index - window_size
        )

        right = min(
            len(sentence),
            center_index + window_size + 1
        )

        for context_index in range(left, right):

            if context_index == center_index:
                continue

            context_word = sentence[context_index]

            training_pairs.append(
                (
                    word_to_id[center_word],
                    word_to_id[context_word]
                )
            )

print("Number of training pairs:", len(training_pairs))

for center, context in training_pairs[:10]:
    print(
        id_to_word[center],
        "->",
        id_to_word[context]
    )

Number of training pairs: 48
the -> cat
cat -> the
cat -> likes
likes -> cat
likes -> milk
milk -> likes
the -> dog
dog -> the
dog -> likes
likes -> dog


这里是把词语变成tensor

In [5]:
center_ids = torch.tensor(
    [pair[0] for pair in training_pairs],
    dtype=torch.long
)

context_ids = torch.tensor(
    [pair[1] for pair in training_pairs],
    dtype=torch.long
)

print(center_ids.shape)
print(context_ids.shape)

torch.Size([48])
torch.Size([48])


建立自己的NLP模型

In [6]:
class MiniWord2Vec(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.output = nn.Linear(
            embedding_dim,
            vocab_size,
            bias=False
        )

    def forward(self, x):

        embedding = self.embedding(x)

        logits = self.output(embedding)

        return logits

In [7]:
embedding_dim = 8

model = MiniWord2Vec(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim
)

print(model)

MiniWord2Vec(
  (embedding): Embedding(12, 8)
  (output): Linear(in_features=8, out_features=12, bias=False)
)


center word ID

 ↓

Embedding

 ↓

8-dimensional vector

↓

Linear

↓

Vocab-size logits

↓

预测 context word

# 来看看没有训练之前猫和狗的向量有多像


In [8]:
def word_similarity(
    model,
    word1,
    word2
):

    embeddings = model.embedding.weight

    vector1 = embeddings[
        word_to_id[word1]
    ]

    vector2 = embeddings[
        word_to_id[word2]
    ]

    similarity = F.cosine_similarity(
        vector1.unsqueeze(0),
        vector2.unsqueeze(0)
    )

    return similarity.item()

In [9]:
before_similarity = word_similarity(
    model,
    "cat",
    "dog"
)

print(
    "Before training:",
    before_similarity
)

Before training: 0.24162966012954712


# week1 最重要的代码

In [10]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.03
)

num_epochs = 500

for epoch in range(num_epochs):

    # 1. 清空旧 gradient
    optimizer.zero_grad()

    # 2. Forward
    logits = model(center_ids)

    # 3. Loss
    loss = F.cross_entropy(
        logits,
        context_ids
    )

    # 4. Backpropagation
    loss.backward()

    # 5. Update parameters
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {loss.item():.4f}"
        )

Epoch   0 | Loss: 2.3960
Epoch 100 | Loss: 1.0200
Epoch 200 | Loss: 1.0187
Epoch 300 | Loss: 1.0183
Epoch 400 | Loss: 1.0182


In [12]:
print("Final loss:", loss.item())

Final loss: 1.0180931091308594


optimizer.zero_grad()

        ↓
清空上一轮 gradient


model(center_ids)

        ↓
Forward propagation


cross_entropy(...)

        ↓
计算预测有多差


loss.backward()

        ↓
计算所有 parameter 的 gradient


optimizer.step()

        ↓
利用 gradient 更新 parameter


In [11]:
after_similarity = word_similarity(
    model,
    "cat",
    "dog"
)

print(
    "Before training:",
    before_similarity
)

print(
    "After training:",
    after_similarity
)

Before training: 0.24162966012954712
After training: 0.8545961976051331
